In [ ]:

# imports
import torch
import shutil
import pickle
import pandas as pd
import numpy as np
from mpmath.libmp import to_int
from torch import nn
from pathlib import Path
from collections import OrderedDict
from kagglehub import dataset_download
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from torch.nn import Sigmoid
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader, WeightedRandomSampler, TensorDataset
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# download dataset

local_data_path = Path("../data")
dataset_base_path = Path(dataset_download("crawford/cat-dataset"))
dataset_base_path = dataset_base_path / "cats"
oreo_path = local_data_path / "oreo"
not_oreo_path = local_data_path / "not_oreo"

oreo_path.mkdir(exist_ok=True)
not_oreo_path.mkdir(exist_ok=True)

for image in local_data_path.glob("*.jpg"):
    shutil.move(str(image), str(oreo_path / image.name))

for subdir in dataset_base_path.iterdir():
    if subdir.is_dir():
        for image in subdir.glob("*.jpg"):
            new_name = f"{subdir.name}_{image.name}"
            shutil.move(str(image), str(not_oreo_path / new_name))

In [3]:
# Split
full_dataset = datasets.ImageFolder(local_data_path)

x_data = []
y_data = []

for image, label in full_dataset:
    x_data.append(image)
    y_data.append(label)

# 80% Train
x_train, x_test, y_train, y_test = train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    stratify=y_data,
    random_state=42
)

# 10% validation
# 10% test
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

KeyboardInterrupt: 

In [ ]:
# Split (faster)

# Apply a basic transform here so we get tensors, not PIL Images
base_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(local_data_path, transform=base_transform)

# Use DataLoader with num_workers for parallel I/O
loader = DataLoader(full_dataset, batch_size=64, shuffle=False, num_workers=4)

x_data_batches = []
y_data_batches = []

for images, labels in loader:
    x_data_batches.append(images)
    y_data_batches.append(labels)

x_data = torch.cat(x_data_batches)       # shape: (N, C, H, W)
y_data = torch.cat(y_data_batches).tolist()

# 80% Train
indices = list(range(len(x_data)))
x_train_idx, x_test_idx, y_train, y_test = train_test_split(
    indices, y_data, test_size=0.2, stratify=y_data, random_state=42
)
x_train = x_data[x_train_idx]
x_test  = x_data[x_test_idx]

# 10% validation / 10% test
train_indices = list(range(len(x_train)))
x_train_idx2, x_val_idx, y_train, y_val = train_test_split(
    train_indices, y_train, test_size=0.2, stratify=y_train, random_state=42
)
x_val   = x_train[x_val_idx]
x_train = x_train[x_train_idx2]

In [21]:
# preprocessing

x_train_proc = x_train.to(device)
x_val_proc = x_val.to(device)
x_test_proc = x_test.to(device)

# Resize, Augment, and Normalize RGB
cnn_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2), # random brightness
])

cnn_x_train_proc = torch.stack([cnn_transform(img) for img in x_train]).to(device)

In [22]:
# Weighted Random Sampler

class_sample_count = np.array([len(np.where(y_train == t)[0]) for t in np.unique(y_train)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in y_train])
samples_weight = torch.from_numpy(samples_weight)

sampler = WeightedRandomSampler(
   weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True
)

#train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=4)
cnn_y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
data = TensorDataset(cnn_x_train_proc, cnn_y_train)
cnn_train_loader = DataLoader(data, batch_size=32, sampler=sampler, num_workers=0)
#val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
#test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

#images, labels = next(iter(train_loader))

In [23]:
# train
x_train_hog = []

for image in x_train_proc:
    img_np = image.cpu().permute(1, 2, 0).numpy()
    feat = hog(
        img_np,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        channel_axis=-1
    )
    x_train_hog.append(feat)
x_train_flat = np.vstack(x_train_hog)

# validation
x_val_hog = []

for image in x_val_proc:
    img_np = image.cpu().permute(1, 2, 0).numpy()
    feat = hog(
        img_np,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        channel_axis=-1
    )
    x_val_hog.append(feat)
x_val_flat = np.vstack(x_val_hog)

# test
x_test_hog = []

for image in x_test_proc:
    img_np = image.cpu().permute(1, 2, 0).numpy()
    feat = hog(
        img_np,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        channel_axis=-1
    )
    x_test_hog.append(feat)
x_test_flat = np.vstack(x_test_hog)

In [25]:
#PCA on the images
pca = PCA(n_components=0.95)
x_train = pca.fit_transform(x_train_flat)
x_val_new = pca.transform(x_val_flat)
x_test_new = pca.transform(x_test_flat)

In [26]:
# standard scaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val_new = scaler.transform(x_val_new)
x_test_new = scaler.transform(x_test_new)

In [9]:
# hyperparameter tuning
C_values = [0.001, 0.01, 0.1, 1]
log_reg_model = None
best_f1 = 0
best_C = None
f1_scores = []
val_accuracy = []

# f1 validation
for C in C_values:
    regression = LogisticRegression(max_iter=3000, solver='saga', C=C, class_weight='balanced')

    regression.fit(x_train, y_train)
    val_preds = regression.predict(x_val_new)

    f1 = f1_score(y_val, val_preds, average='macro')
    f1_scores.append(f1)
    accuracy = accuracy_score(y_val, val_preds)
    val_accuracy.append(accuracy)

    print(f"C={C}, Validation F1={f1}, Validation Accuracy: {accuracy}")
    if f1 > best_f1:
        best_f1 = f1
        log_reg_model = regression
        best_C = C

print(f"\nBest C: {best_C} with F1={best_f1}")

predictions = log_reg_model.predict(x_test_new)
print(f"Predictions: {predictions}")
print(f"Actual labels: {y_test}")
print(f"Accuracy: {accuracy_score(y_test, predictions)}")

C:\Users\tdows\Desktop\Machine Learning\Machine_Learning_OreoFinder\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C=0.001, Validation F1=0.5530316544766208, Validation Accuracy: 0.8658536585365854


C:\Users\tdows\Desktop\Machine Learning\Machine_Learning_OreoFinder\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C=0.01, Validation F1=0.5798356612979637, Validation Accuracy: 0.8908536585365854


C:\Users\tdows\Desktop\Machine Learning\Machine_Learning_OreoFinder\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C=0.1, Validation F1=0.5719311518819746, Validation Accuracy: 0.9158536585365854
C=1, Validation F1=0.5706899232587571, Validation Accuracy: 0.9189024390243903

Best C: 0.01 with F1=0.5798356612979637
Predictions: [0 0 0 ... 0 0 0]
Actual labels: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [10]:
# K-NN

# Validation
k_val = [1, 2, 3, 5, 7, 9]
best_k = None
best_f1 = 0
for k in k_val:
    model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    model.fit(x_train, y_train)
    val_predictions = model.predict(x_val_new)
    f1 = f1_score(y_val, val_predictions, average='macro')
    print(f"k={k}, Validation F1 Score: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_k = k

# Training with best k
print(f"Best k: {best_k} with F1 Score: {best_f1:.4f}")
knn_model = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn_model.fit(x_train, y_train)

# Printing predictions and actual labels
predictions = knn_model.predict(x_test_new)
for prediction in predictions:
    print(f"{prediction}")

print(f"Actual labels: {y_test}")

accuracy = np.mean(predictions == y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Showing incorrect predictions and their neighbors
incorrect = np.where(((predictions == 1) & (y_test == 0)) | ((predictions == 0) & (y_test == 1)))[0]
num_mistakes = len(incorrect)
if num_mistakes > 0:
    fig, axes = plt.subplots(num_mistakes, best_k + 1, figsize=(20, 4 * num_mistakes))

    if num_mistakes == 1:
        axes = np.expand_dims(axes, axis=0)
    for i, test_idx in enumerate(incorrect):
        sample_image = x_test_new[test_idx]
        distances,  neighbor_indices = knn_model.kneighbors(sample_image.reshape(1, -1), n_neighbors=best_k)
        ax_test = axes[i, 0]
        img_reshaped = original_test_images[test_idx].permute(1, 2, 0).numpy()
        ax_test.imshow(img_reshaped)
        ax_test.set_title(f"INCORRECT\n(Index {test_idx})", color='red')
        ax_test.axis('off')
        for j, train_idx in enumerate(neighbor_indices[0]):
            ax_nb = axes[i, j + 1]
            neighbor_img = x_train[train_idx].reshape(3, 128, 128).transpose(1, 2, 0)
            ax_nb.imshow(neighbor_img)
            ax_nb.set_title(f"Neighbor {j+1}\nDist: {distances[0][j]:.2f}")
            ax_nb.axis('off')

plt.tight_layout()
plt.show()


k=1, Validation F1 Score: 0.5727
k=2, Validation F1 Score: 0.4938
k=3, Validation F1 Score: 0.5393
k=5, Validation F1 Score: 0.5418
k=7, Validation F1 Score: 0.4938
k=9, Validation F1 Score: 0.4938
Best k: 1 with F1 Score: 0.5727
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
0
0
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
0
0
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
1
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


<Figure size 640x480 with 0 Axes>

In [11]:
  # CNN
learning_rate = 0.0001
epochs = 20

cnn_model = nn.Sequential(
    OrderedDict([
        ("conv1", nn.Conv2d(3, 128, kernel_size=3, padding=1)),
        ("norm1", nn.BatchNorm2d(128)),
        ("relu1", nn.LeakyReLU(0.01)),
        ("pool1", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("conv2", nn.Conv2d(128, 64, kernel_size=3, padding=1)),
        ("norm2", nn.BatchNorm2d(64)),
        ("relu2", nn.LeakyReLU(0.01)),
        ("pool2", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("conv3", nn.Conv2d(64, 32, kernel_size=3, padding=1)),
        ("norm3", nn.BatchNorm2d(32)),
        ("relu3", nn.LeakyReLU(0.01)),
        ("pool3", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("dropout", nn.Dropout2d(0.25)),
        ("flatten", nn.Flatten()),
        ("linear1", nn.Linear(32 * 16 * 16, 128)),
        ("relu5", nn.ReLU()),
        ("linear2", nn.Linear(in_features=128, out_features=1)),
    ])
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    cnn_model.train()
    total_loss = 0

    for images, labels in cnn_train_loader:
        optimizer.zero_grad()

        outputs = cnn_model(images).squeeze(1)

        loss = criterion(outputs, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
    print(f"epoch {epoch + 1}/{epochs}")
    print(f"loss {total_loss:.4f}")

epoch 1/20
loss 51.6195
epoch 2/20
loss 16.5691
epoch 3/20
loss 9.0748
epoch 4/20
loss 7.0299
epoch 5/20
loss 4.9445
epoch 6/20
loss 4.1423
epoch 7/20
loss 2.3730
epoch 8/20
loss 1.1931
epoch 9/20
loss 1.0491
epoch 10/20
loss 1.2158
epoch 11/20
loss 0.8455
epoch 12/20
loss 0.9912
epoch 13/20
loss 2.1091
epoch 14/20
loss 2.7046
epoch 15/20
loss 2.5313
epoch 16/20
loss 0.8015
epoch 17/20
loss 1.7061
epoch 18/20
loss 0.9589
epoch 19/20
loss 1.1836
epoch 20/20
loss 2.0056


In [12]:
# Save the models


log_reg_path = Path("..") / "models" / "logRegModel.pkl"
with open(log_reg_path, 'wb') as f:
    pickle.dump(log_reg_model, f)

knn_path = Path("..") / "models" / "knnModel.pkl"
with open(knn_path, 'wb') as f:
    pickle.dump(knn_model, f)

cnn_path = Path("..") / "models" / "cnnModel.pkl"
with open(cnn_path, 'wb') as f:
    pickle.dump(cnn_model, f)

In [8]:
# Load the models

log_reg_path = Path("..") / "models" / "logRegModel.pkl"
with open(log_reg_path, 'rb') as f:
    log_reg_model = pickle.load(f)

knn_path = Path("..") / "models" / "knnModel.pkl"
with open(knn_path, 'rb') as f:
    knn_model = pickle.load(f)

cnn_path = Path("..") / "models" / "cnnModel.pkl"
with open(cnn_path, 'rb') as f:
    cnn_model = pickle.load(f)

In [9]:
# evaluation

models = pd.DataFrame(index=["Accuracy", "Precision", "Recall", "F1", "AUC"])

lr_accuracy = accuracy_score(y_test, log_reg_model.predict(x_test_new))
lr_precision = precision_score(y_test, log_reg_model.predict(x_test_new))
lr_recall = recall_score(y_test, log_reg_model.predict(x_test_new))
lr_f1 = f1_score(y_test, log_reg_model.predict(x_test_new))
lr_auc = roc_auc_score(y_test, log_reg_model.predict(x_test_new))
lr_roc = roc_curve(y_test, log_reg_model.predict(x_test_new))

models["Logistic Regression"] = [lr_accuracy, lr_precision, lr_recall, lr_f1, lr_auc]

knn_accuracy = accuracy_score(y_test, knn_model.predict(x_test_new))
knn_precision = precision_score(y_test, knn_model.predict(x_test_new))
knn_recall = recall_score(y_test, knn_model.predict(x_test_new))
knn_f1 = f1_score(y_test, knn_model.predict(x_test_new))
knn_auc = roc_auc_score(y_test, knn_model.predict(x_test_new))
knn_roc = roc_curve(y_test, knn_model.predict(x_test_new))

models["K-Nearest Neighbor"] = [knn_accuracy, knn_precision, knn_recall, knn_f1, knn_auc]

predictions = []

cnn_model.eval()
with torch.no_grad():
    for image in x_test:
        probability = torch.sigmoid(cnn_model(image))
        prediction = (probability > 0.5).int.item()

        predictions.append(prediction)

cnn_accuracy = accuracy_score(y_test, predictions)
cnn_precision = precision_score(y_test, predictions)
cnn_recall = recall_score(y_test, predictions)
cnn_f1 = f1_score(y_test, predictions)
cnn_auc = roc_auc_score(y_test, predictions)
cnn_roc = roc_curve(y_test, predictions)

models["Convolutional Neural Network"] = [cnn_accuracy, cnn_precision, cnn_recall, cnn_f1, cnn_auc]

TypeError: conv2d() received an invalid combination of arguments - got (Image, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!Image!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!Image!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
